## 01 · Extract data

Reads five of the raw files in `data/raw/`, keeps only the series the project needs, standardises the dates, and saves one CSV per source to `data/processed/`. The USGS production data is used later, for the index weights.

| Source | File | Series |
|---|---|---|
| World Bank Pink Sheet | `worldbank_pinksheet_monthly.xlsx` | metal prices, Brent oil, natural gas |
| New York Fed | `nyfed_gscpi.xls` | supply chain pressure (GSCPI) |
| CPB World Trade Monitor | `cpb_world_trade_monitor_2026m07.xlsx` | industrial production |
| BIS via FRED | `bis_usd_broad_neer_fred.xlsx` | US dollar index |
| IMF | `imf_pcps_metals_monthly.csv` | cobalt, lithium |

---

*Code written with help of Claude (Anthropic), asked to produce code that extracts the needed series from my raw data files and summarises their coverage.*

In [37]:
from pathlib import Path
import pandas as pd
import numpy as np

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(exist_ok=True)

def to_month(dates):
    text = pd.Series(dates).astype(str).str.replace("M", "-").str.replace("m", "-")
    return pd.to_datetime(text, errors="coerce").dt.to_period("M").dt.to_timestamp().values

In [ ]:
### WorldBank prices database
PINK_columns = {
    "Copper": "copper", "Aluminum": "aluminium", "Nickel": "nickel", "Tin": "tin",
    "Zinc": "zinc", "Lead": "lead",
    "Crude oil, Brent": "brent", "Natural gas index": "naturalgas_index",
}

pink = pd.read_excel(RAW / "worldbank_pinksheet_monthly.xlsx", sheet_name="Monthly Prices", skiprows=4)
pink = pink.iloc[1:]                                   

prices = pink[list(PINK_columns)].rename(columns=PINK_columns)
prices = prices.apply(pd.to_numeric, errors="coerce")  
prices.insert(0, "date", to_month(pink.iloc[:, 0]))

In [ ]:
### GSCPI data 
gscpi = pd.read_excel(RAW / "nyfed_gscpi.xls", sheet_name="GSCPI Monthly Data",
                      skiprows=5, header=None, usecols=[0, 1], names=["date", "gscpi"])
gscpi["date"] = to_month(pd.to_datetime(gscpi["date"], format="%d-%b-%Y"))

In [ ]:
### CPB industrial production data
CPB_codes = {"ipz_w1_qnmi_sp": "ind_production_world", "ipz_w1_qnmi_sm": "ind_production_world_importweighted", "ipz_cn_qnmi_sp": "ind_production_china"}

cpb = pd.read_excel(RAW / "cpb_world_trade_monitor_2026m07.xlsx", sheet_name="inpro_out", header=None)
date_cols = cpb.columns[cpb.iloc[3].astype(str).str.match(r"\d{4}m\d{2}")]   # 

industrial_production = cpb.set_index(2).loc[list(CPB_codes), date_cols].T   
industrial_production.columns = [CPB_codes[c] for c in industrial_production.columns]
industrial_production = industrial_production.astype(float).reset_index(drop=True)
industrial_production.insert(0, "date", to_month(cpb.loc[3, date_cols]))

In [ ]:
### dollar index: the US dollar's value against a broad basket of currencies. 'NEER' stands for nominal effective exchange rate. Higher = stronger US dollar. 
bis = pd.read_excel(RAW / "bis_usd_broad_neer_fred.xlsx", sheet_name="Monthly", header=None)
start = bis.index[bis[0] == "observation_date"][0] + 1    

usd = pd.DataFrame({
    "date": to_month(bis.iloc[start:, 0]),
    "us_dollar_neer": pd.to_numeric(bis.iloc[start:, 1]).values,
})

In [ ]:
### IMF prices 
IMF_codenames = {
    "G001.PCOBA.USD.M": "cobalt",
    "G001.PLITH.USD.M": "lithium",
    "G001.PCOPP.USD.M": "copper_imf",
    "G001.PALUM.USD.M": "aluminium_imf",
    "G001.PNICK.USD.M": "nickel_imf",
    "G001.PTIN.USD.M":  "tin_imf",
}

imf_raw = pd.read_csv(RAW / "imf_pcps_metals_monthly.csv")
imf_raw = imf_raw.set_index("SERIES_CODE").loc[list(IMF_codenames)]   

date_cols = [c for c in imf_raw.columns if c[:4].isdigit()]      
imf = imf_raw[date_cols].T                                         
imf.columns = [IMF_codenames[c] for c in imf.columns]                  

imf.index = pd.to_datetime(imf.index.str.replace("-M", "-") + "-01")   
imf = imf.apply(pd.to_numeric, errors="coerce").rename_axis("date").reset_index()

In [ ]:
### save data as .csv files 
outputs = {"prices.csv": prices, "gscpi.csv": gscpi,
           "industrial_production.csv": industrial_production, "usd.csv": usd,
           "prices_imf.csv": imf}
for file, df in outputs.items():
    df.to_csv(PROCESSED / file, index=False)
    print(f"saved {file:28s} {len(df):5d} rows")

saved prices.csv                     800 rows
saved gscpi.csv                      344 rows
saved industrial_production.csv      319 rows
saved usd.csv                        391 rows
saved prices_imf.csv                 333 rows


In [45]:
### short descriptive analysis
rows = []
for file, df in outputs.items():
    for col in df.columns[1:]:
        s = df.dropna(subset=[col])
        months = pd.date_range(s["date"].min(), s["date"].max(), freq="MS")
        rows.append([file, col, s["date"].min().strftime("%Y-%m"), s["date"].max().strftime("%Y-%m"),
                     len(s), len(months) - len(s)])

overview = pd.DataFrame(rows, columns=["file", "series", "first_date", "last_date", "number_of_rows", "NA"])
overview

,file,series,first_date,last_date,number_of_rows,NA
0,prices.csv,copper,1960-01,2026-08,800,0
1,prices.csv,aluminium,1960-01,2026-08,800,0
2,prices.csv,nickel,1960-01,2026-08,800,0
3,prices.csv,tin,1960-01,2026-08,800,0
4,prices.csv,zinc,1960-01,2026-08,800,0
5,prices.csv,lead,1960-01,2026-08,800,0
6,prices.csv,brent,1960-01,2026-08,800,0
7,prices.csv,naturalgas_index,1977-01,2026-08,596,0
8,gscpi.csv,gscpi,1998-01,2026-08,344,0
9,industrial_production.csv,ind_production_world,2000-01,2026-07,319,0
